# exp034_rerank_a023_0_4_a456_0_2 — specialist rerank inference

**Base ensemble:** exp025_multiseed_ensemble (4 models)
**Specialists:** exp034a_023_specialist (α=0.4), exp034b_456_specialist (α=0.2)

Inference pipeline: average base model probs → soft specialist rerank → threshold tuning offsets → submission.csv
Runtime: ~15–20 min on GPU T4.

## Setup checklist
1. **Accelerator**: GPU T4×1
2. **Internet**: Off
3. **Competition dataset**: attach `spr-2026-mammography-report-classification`
4. **Model dataset**: upload all 6 model dirs + thresholds.npy as one Kaggle dataset
   (see the prep script output for exact directory layout)


In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'sentencepiece>=0.1.99',
    'pyyaml>=6.0', 'pandas>=2.0', 'numpy>=1.24',
])


In [ ]:
import os
os.makedirs('src/models', exist_ok=True)


In [ ]:
open('src/__init__.py', 'w').close()  # empty init file


In [ ]:
%%writefile src/data.py
"""
Data loading and fold creation.
Handles both local and Kaggle runtime paths automatically.
"""
import os
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

# Searched in order - first match wins
_DATA_DIRS = [
    "/kaggle/input/competitions/spr-2026-mammography-report-classification",
    "/kaggle/input/spr-2026-mammography-report-classification",
    "./spr-2026-mammography-report-classification",
    "../spr-2026-mammography-report-classification",
    "../../spr-2026-mammography-report-classification",
]


def find_data_dir() -> str:
    env_data_dir = os.getenv("SPR2026_DATA_DIR")
    if env_data_dir:
        env_path = Path(env_data_dir)
        if (env_path / "train.csv").exists():
            return str(env_path)

    tried = []
    for data_dir in _DATA_DIRS:
        train_path = Path(data_dir) / "train.csv"
        tried.append(str(train_path))
        if train_path.exists():
            return str(Path(data_dir))

    raise FileNotFoundError(
        "Could not find data directory. Checked these train.csv paths:\n"
        + "\n".join(f"- {path}" for path in tried)
        + "\nSet SPR2026_DATA_DIR or place the competition data in "
        "./spr-2026-mammography-report-classification/."
    )


def load_train(data_dir: str = None) -> pd.DataFrame:
    if data_dir is None:
        data_dir = find_data_dir()
    df = pd.read_csv(Path(data_dir) / "train.csv")
    df.columns = df.columns.str.strip()
    df["target"] = df["target"].astype(int)
    return df


def load_test(data_dir: str = None) -> pd.DataFrame:
    if data_dir is None:
        data_dir = find_data_dir()
    path = Path(data_dir) / "test.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"test.csv not found at {path}.\n"
            "This is expected locally - it only exists in the Kaggle evaluation runtime."
        )
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df


def make_folds(df: pd.DataFrame, n_folds: int = 5, seed: int = 42,
               group_aware: bool = True) -> pd.DataFrame:
    """Add a 'fold' column (0..n_folds-1).

    group_aware=True (default): identical report texts are guaranteed to land in
    the same fold, eliminating leakage from the ~54% duplicate rows.
    Stratification is on the majority label per text group.
    """
    df = df.copy().reset_index(drop=True)
    df["fold"] = -1

    if group_aware:
        unique_texts = df["report"].unique()
        text_to_gid  = {t: i for i, t in enumerate(unique_texts)}
        groups       = df["report"].map(text_to_gid).values

        # Stratify on majority label per group (handles 11 conflicting-label groups)
        majority_label = df.groupby("report")["target"].agg(lambda x: x.mode()[0])
        strat_y        = df["report"].map(majority_label).values

        sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        for fold, (_, val_idx) in enumerate(sgkf.split(df, strat_y, groups)):
            df.loc[val_idx, "fold"] = fold
    else:
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        for fold, (_, val_idx) in enumerate(skf.split(df, df["target"])):
            df.loc[val_idx, "fold"] = fold

    return df


def load_synthetic(classes: list = None, data_dir: str = None) -> pd.DataFrame:
    """Load synthetic mammography reports from the competition's external dataset.

    Args:
        classes: list of int target classes to keep (e.g. [0,3,4,5,6]).
                 None means return all classes.
        data_dir: override data directory; defaults to auto-detected competition dir.

    Returns DataFrame with columns: report, target, text (pre-cleaned).
    """
    if data_dir is None:
        data_dir = find_data_dir()
    synth_path = Path(data_dir) / "synthetic_ext_data" / "mammography_reports_pt_full.csv"
    if not synth_path.exists():
        raise FileNotFoundError(f"Synthetic data not found at {synth_path}")
    df = pd.read_csv(synth_path)
    df.columns = df.columns.str.strip()
    df["target"] = df["target"].astype(int)
    if classes is not None:
        df = df[df["target"].isin(classes)].reset_index(drop=True)
    # Use the full report field to match real training data format
    return df[["report", "target"]].copy()


def dedup_for_training(df: pd.DataFrame) -> pd.DataFrame:
    """Return a deduplicated DataFrame for use as training data.

    Keeps one representative row per unique report text. For groups with
    conflicting labels (11 groups, all with an overwhelming majority), replaces
    the label with the majority vote. The 'fold' column is preserved — group-aware
    folds guarantee all copies of a text share the same fold, so the representative
    row's fold is correct.
    """
    df = df.copy()

    # Fix minority labels in the 11 conflicting groups
    majority_label = df.groupby("report")["target"].agg(lambda x: x.mode()[0])
    df["target"]   = df["report"].map(majority_label)

    dedup = (
        df.drop_duplicates(subset="report")
          .reset_index(drop=False)
          .rename(columns={"index": "orig_idx"})
    )
    return dedup


In [ ]:
%%writefile src/features.py
"""
Text preprocessing and TF-IDF feature extraction for Portuguese medical reports.
"""
import pickle
import re
import os
from typing import List, Tuple, Optional

import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer


def clean_text(text: str) -> str:
    """
    Minimal cleaning that preserves medically meaningful tokens.
    - Keeps <DATA> placeholder (correlates with comparative exams → BI-RADS 2/3)
    - Normalises whitespace; lowercases
    - Does NOT strip accents — Portuguese accents carry lexical meaning
    """
    if not isinstance(text, str):
        text = str(text)
    # Collapse newlines / tabs to a single space
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r" {2,}", " ", text).strip()
    text = text.lower()
    return text


def _make_vectorizer(cfg: dict) -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer=cfg.get("analyzer", "word"),
        ngram_range=tuple(cfg.get("ngram_range", [1, 2])),
        max_features=cfg.get("max_features", 150_000),
        sublinear_tf=cfg.get("sublinear_tf", True),
        min_df=cfg.get("min_df", 2),
        strip_accents=None,  # preserve Portuguese characters
    )


def build_features(
    texts_train: List[str],
    texts_val: Optional[List[str]] = None,
    config: Optional[dict] = None,
) -> Tuple:
    """
    Fit TF-IDF (word + char) on texts_train.
    Returns (X_train, vectorizers) or (X_train, X_val, vectorizers).
    """
    cfg = config or {}
    word_cfg = cfg.get("word_tfidf", {})
    char_cfg  = cfg.get("char_tfidf", {})

    # Default char config if not provided
    if "analyzer" not in char_cfg:
        char_cfg = {"analyzer": "char_wb", "ngram_range": [2, 5],
                    "max_features": 150_000, "sublinear_tf": True, "min_df": 3}

    word_vec = _make_vectorizer(word_cfg)
    char_vec  = _make_vectorizer(char_cfg)

    X_word_train = word_vec.fit_transform(texts_train)
    X_char_train  = char_vec.fit_transform(texts_train)
    X_train = hstack([X_word_train, X_char_train], format="csr")

    vectorizers = (word_vec, char_vec)

    if texts_val is not None:
        X_word_val = word_vec.transform(texts_val)
        X_char_val  = char_vec.transform(texts_val)
        X_val = hstack([X_word_val, X_char_val], format="csr")
        return X_train, X_val, vectorizers

    return X_train, vectorizers


def transform_features(texts: List[str], vectorizers: Tuple) -> csr_matrix:
    word_vec, char_vec = vectorizers
    X_word = word_vec.transform(texts)
    X_char  = char_vec.transform(texts)
    return hstack([X_word, X_char], format="csr")


def save_vectorizers(vectorizers: Tuple, out_dir: str) -> None:
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "vectorizers.pkl"), "wb") as f:
        pickle.dump(vectorizers, f)


def load_vectorizers(out_dir: str) -> Tuple:
    with open(os.path.join(out_dir, "vectorizers.pkl"), "rb") as f:
        return pickle.load(f)


In [ ]:
%%writefile src/evaluate.py
"""
Metrics, reporting, and the master results log.
Always report macro F1 + per-class breakdown — never accuracy alone.
"""
import json
import os

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score

CLASSES = [0, 1, 2, 3, 4, 5, 6]
RESULTS_LOG = "experiments/results.csv"

# Columns shown in --compare (focused on rare/hard classes)
_COMPARE_COLS = [
    "experiment", "macro_f1",
    "f1_c0", "f1_c1", "f1_c3", "f1_c4", "f1_c5", "f1_c6",
    "duration_s", "timestamp", "notes",
]


def compute_metrics(y_true, y_pred, labels=None) -> dict:
    labels = CLASSES if labels is None else list(labels)
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)
    report = classification_report(
        y_true, y_pred, labels=labels, zero_division=0, output_dict=True
    )
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return {"macro_f1": macro_f1, "report": report, "confusion_matrix": cm.tolist(), "labels": labels}


def print_metrics(metrics: dict, title: str = "", labels=None) -> None:
    labels = metrics.get("labels", CLASSES) if labels is None else list(labels)
    if title:
        print(f"\n{'='*55}\n{title}\n{'='*55}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}\n")
    print(f"{'Class':<8} {'F1':>6} {'Prec':>6} {'Rec':>6} {'N':>6}")
    print("-" * 38)
    for cls in labels:
        r = metrics["report"].get(str(cls), {})
        f1  = r.get("f1-score",  0.0)
        pre = r.get("precision", 0.0)
        rec = r.get("recall",    0.0)
        n   = int(r.get("support", 0))
        print(f"{cls:<8} {f1:>6.3f} {pre:>6.3f} {rec:>6.3f} {n:>6}")
    print()


def save_metrics(metrics: dict, out_dir: str) -> None:
    if out_dir.endswith(".json"):
        path = out_dir
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    else:
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, "metrics.json")
    with open(path, "w") as f:
        json.dump(metrics, f, indent=2)


def append_to_results_log(
    exp_name: str,
    metrics: dict,
    config: dict,
    timestamp: str = "",
    duration: float = 0.0,
    notes: str = "",
) -> None:
    """Upsert one row into experiments/results.csv."""
    row = {
        "experiment":  exp_name,
        "macro_f1":    round(metrics["macro_f1"], 4),
        "model_type":  config.get("model", {}).get("type", ""),
        "n_folds":     config.get("data", {}).get("n_folds", ""),
        "seed":        config.get("seed", ""),
        "timestamp":   timestamp,
        "duration_s":  round(duration, 1),
        "notes":       notes or config.get("notes", ""),
    }
    for cls in CLASSES:
        r = metrics["report"].get(str(cls), {})
        row[f"f1_c{cls}"] = round(r.get("f1-score", 0.0), 4)

    df_row = pd.DataFrame([row])
    if os.path.exists(RESULTS_LOG):
        existing = pd.read_csv(RESULTS_LOG, keep_default_na=False)
        existing = existing[existing["experiment"] != exp_name]
        df_out = pd.concat([existing, df_row], ignore_index=True)
    else:
        os.makedirs(os.path.dirname(RESULTS_LOG), exist_ok=True)
        df_out = df_row

    df_out = df_out.sort_values("macro_f1", ascending=False)
    df_out.to_csv(RESULTS_LOG, index=False)
    print(f"Results logged → {RESULTS_LOG}")


def print_comparison(full: bool = False) -> None:
    """Print the leaderboard table. full=True shows all columns."""
    if not os.path.exists(RESULTS_LOG):
        print("No experiments logged yet.")
        return

    df = (pd.read_csv(RESULTS_LOG, keep_default_na=False)
            .sort_values("macro_f1", ascending=False)
            .reset_index(drop=True))
    df.index += 1  # rank starts at 1

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 140)
    pd.set_option("display.float_format", "{:.4f}".format)

    print(f"\n{'='*55}")
    print(f"  Experiment leaderboard  (OOF macro F1, descending)")
    print(f"{'='*55}")

    if full:
        print(df.to_string())
    else:
        cols = [c for c in _COMPARE_COLS if c in df.columns]
        print(df[cols].to_string())

    # Highlight the improvement vs the row below
    if len(df) > 1:
        best = df["macro_f1"].iloc[0]
        second = df["macro_f1"].iloc[1]
        print(f"\n  Best vs 2nd: +{best - second:+.4f}  ({df['experiment'].iloc[0]})")
    print()


In [ ]:
%%writefile src/threshold.py
"""
Post-processing: per-class decision threshold tuning on OOF probabilities.
Tuning is explicitly allowed by competition rules. Always tune on OOF only.
"""
import numpy as np
from sklearn.metrics import f1_score

CLASSES = [0, 1, 2, 3, 4, 5, 6]


def tune_thresholds(
    oof_probs: np.ndarray,
    oof_labels: np.ndarray,
    n_iter: int = 500,
    seed: int = 42,
) -> np.ndarray:
    """
    Random-search for per-class additive offsets that maximise OOF macro F1.
    Returns an offset array of shape (n_classes,).
    Apply at inference time: preds = argmax(probs + offsets, axis=1).
    """
    baseline_preds = np.argmax(oof_probs, axis=1)
    best_f1 = f1_score(
        oof_labels, baseline_preds, average="macro", labels=CLASSES, zero_division=0
    )
    best_offsets = np.zeros(len(CLASSES))
    print(f"Threshold tuning baseline macro F1: {best_f1:.4f}")

    rng = np.random.RandomState(seed)
    for _ in range(n_iter):
        offsets = rng.uniform(-0.3, 0.3, size=len(CLASSES))
        preds = np.argmax(oof_probs + offsets, axis=1)
        f1 = f1_score(oof_labels, preds, average="macro", labels=CLASSES, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_offsets = offsets.copy()

    print(f"Threshold tuning best macro F1:     {best_f1:.4f}")
    print(f"Offsets: {dict(zip(CLASSES, best_offsets.round(4)))}")
    return best_offsets


def apply_thresholds(probs: np.ndarray, offsets: np.ndarray) -> np.ndarray:
    return np.argmax(probs + offsets, axis=1)


In [ ]:
%%writefile src/logging_utils.py
"""
Dual-output logging: every print() goes to both the terminal and a log file.
tqdm writes to stderr so it stays on the terminal only — logs stay clean.
"""
import io
import os
import re
import sys
from contextlib import contextmanager
from datetime import datetime


# Strip ANSI escape codes (colour, cursor movement) before writing to file
_ANSI_RE = re.compile(r"\x1b\[[0-9;]*[mABCDEFGHJKSTfhilmnsu]|\r")


class _Tee(io.TextIOBase):
    """Writes to both the original stdout and a log file simultaneously."""

    def __init__(self, log_path: str, original_stdout):
        self._stdout = original_stdout
        self._file = open(log_path, "a", encoding="utf-8", buffering=1)

    def write(self, text: str) -> int:
        self._stdout.write(text)
        self._file.write(_ANSI_RE.sub("", text))
        return len(text)

    def flush(self):
        self._stdout.flush()
        self._file.flush()

    def close(self):
        self._file.close()

    # Needed so tqdm and other libs can check isatty on the underlying stream
    def isatty(self) -> bool:
        return self._stdout.isatty()

    @property
    def encoding(self):
        return self._stdout.encoding


@contextmanager
def run_log(out_dir: str, filename: str = "train.log"):
    """
    Context manager that tees all print() output to out_dir/train.log.
    Usage:
        with run_log(out_dir):
            ... training code ...
    """
    os.makedirs(out_dir, exist_ok=True)
    log_path = os.path.join(out_dir, filename)

    tee = _Tee(log_path, sys.stdout)
    original_stdout = sys.stdout
    sys.stdout = tee
    try:
        print(f"[log] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  →  {log_path}")
        yield log_path
    finally:
        sys.stdout = original_stdout
        tee.close()


In [ ]:
%%writefile src/models/__init__.py
"""
Model dispatcher — routes build/save/load to the right backend by model type.
Add new model families here; train.py and predict.py import only from src.models.
"""
import os
import pickle

LINEAR_TYPES = {"logistic_regression", "linear_svc"}
GBM_TYPES = {"lgbm"}


def build_model(config: dict):
    model_type = config.get("type", "logistic_regression")
    if model_type in LINEAR_TYPES:
        from src.models.linear import build_model as _build
        return _build(config)
    if model_type in GBM_TYPES:
        from src.models.gbm import build_model as _build
        return _build(config)
    raise ValueError(
        f"Unknown model type: '{model_type}'. "
        f"Choose from: {sorted(LINEAR_TYPES | GBM_TYPES)}"
    )


def save_model(model, out_dir: str) -> None:
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "model.pkl"), "wb") as f:
        pickle.dump(model, f)


def load_model(out_dir: str):
    with open(os.path.join(out_dir, "model.pkl"), "rb") as f:
        return pickle.load(f)


In [ ]:
%%writefile src/models/transformer.py
"""
HuggingFace transformer model helpers — build / save / load.
Actual training is handled by src/train_transformer.py.
"""
import os

TRANSFORMER_TYPES = {"transformer", "bert", "xlmr"}
N_CLASSES = 7


def build_model(config: dict):
    """Return (model, tokenizer) initialised from a pretrained checkpoint."""
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    pretrained = config.get("pretrained", "neuralmind/bert-base-portuguese-cased")
    num_labels = int(config.get("num_labels", N_CLASSES))
    tokenizer = AutoTokenizer.from_pretrained(pretrained)
    model = AutoModelForSequenceClassification.from_pretrained(
        pretrained,
        num_labels=num_labels,
        ignore_mismatched_sizes=True,
    )
    return model, tokenizer


def save_model(model, tokenizer, out_dir: str) -> None:
    model_dir = os.path.join(out_dir, "model")
    os.makedirs(model_dir, exist_ok=True)
    # Unwrap DataParallel before saving
    m = model.module if hasattr(model, "module") else model
    m.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)


def load_model(out_dir: str):
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    model_dir = os.path.join(out_dir, "model")
    # local_files_only=True prevents HuggingFace from validating the path as a
    # repo_id, which fails on deep local paths with multiple slashes (Kaggle).
    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir, local_files_only=True
    )
    return model, tokenizer


In [ ]:
%%writefile src/train_transformer.py
"""
GPU-accelerated transformer fine-tuning with OOF cross-validation.
Supports BERT-style models via HuggingFace transformers.

Features:
  - Mixed precision (torch.amp FP16)
  - Multi-GPU via DataParallel (uses all visible GPUs automatically)
  - Gradient accumulation
  - Linear warmup + linear decay LR schedule
  - Class-weighted cross-entropy (handles the 87% class-2 imbalance)
  - Best-epoch checkpointing within each fold
  - Weights & Biases tracking (optional — set wandb.enabled: true in config)
  - Same output structure as train.py (oof_preds.csv, metrics.json, etc.)
"""
import gc
import json
import os
import re
import shutil
import subprocess
import time
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from transformers import get_linear_schedule_with_warmup
import yaml
from tqdm import tqdm

from src.data import dedup_for_training, load_synthetic, load_train, make_folds
from src.evaluate import append_to_results_log, compute_metrics, print_metrics, save_metrics
from src.features import clean_text
from src.logging_utils import run_log
from src.models.transformer import build_model, save_model
from src.threshold import tune_thresholds

CLASSES = [0, 1, 2, 3, 4, 5, 6]
N_CLASSES = len(CLASSES)


class _FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets, reduction: str = "mean"):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        if reduction == "none":
            return loss
        return loss.mean()


# ── Device setup ─────────────────────────────────────────

def _setup_device(gpu_cfg=None):
    """
    Resolve which GPU(s) to use from the top-level 'gpu' config field.

    Config examples:
      gpu: all      → all available GPUs (DataParallel)
      gpu: 0        → GPU 0 only
      gpu: 1        → GPU 1 only
      gpu: [0, 1]   → both GPUs explicitly

    You can also override at the shell level with CUDA_VISIBLE_DEVICES:
      CUDA_VISIBLE_DEVICES=1 python run.py configs/exp004.yaml --train
    """
    if not torch.cuda.is_available():
        print("No GPU available — running on CPU")
        return torch.device("cpu"), []

    n_available = torch.cuda.device_count()

    if gpu_cfg is None or gpu_cfg == "all":
        gpu_ids = list(range(n_available))
    elif isinstance(gpu_cfg, int):
        gpu_ids = [gpu_cfg]
    elif isinstance(gpu_cfg, list):
        gpu_ids = [int(g) for g in gpu_cfg]
    else:
        raise ValueError(
            f"Invalid 'gpu' config value: {gpu_cfg!r}. "
            "Use an integer, a list of integers, or 'all'."
        )

    for g in gpu_ids:
        if g >= n_available:
            raise ValueError(
                f"GPU {g} requested but only {n_available} GPU(s) available "
                f"(indices 0–{n_available - 1})."
            )

    label = " + ".join(
        f"[{g}] {torch.cuda.get_device_name(g)}" for g in gpu_ids
    )
    print(f"GPUs : {label}")
    return torch.device(f"cuda:{gpu_ids[0]}"), gpu_ids


# ── Weights & Biases ──────────────────────────────────────

def _init_wandb(config, exp_name):
    """
    Initialise a wandb run and return it, or return None if wandb is
    disabled in config or not installed.
    """
    wb = config.get("wandb", {})
    if not wb.get("enabled", False):
        return None
    try:
        import wandb
        mdl = config["model"]
        run = wandb.init(
            project=wb.get("project", "spr-2026-mammography"),
            entity=wb.get("entity") or None,
            name=exp_name,
            tags=wb.get("tags", []),
            config={
                "experiment":   exp_name,
                "pretrained":   mdl["pretrained"],
                "max_length":   mdl.get("max_length", 512),
                "n_folds":      config["data"]["n_folds"],
                "seed":         config.get("seed", 42),
                **mdl.get("params", {}),
            },
            reinit=True,
        )
        # Custom x-axis so per-fold/epoch charts align correctly
        wandb.define_metric("epoch")
        wandb.define_metric("train/*", step_metric="epoch")
        wandb.define_metric("val/*",   step_metric="epoch")
        print(f"wandb run : {run.url}")
        return run
    except Exception as exc:
        print(f"wandb init failed ({exc}) — continuing without tracking")
        return None


def _wandb_log_epoch(run, fold, epoch, train_loss, ep_metrics, lr):
    if run is None:
        return
    import wandb
    log = {
        "epoch":      epoch,
        "fold":       fold,
        "train/loss": train_loss,
        "val/macro_f1": ep_metrics["macro_f1"],
        "lr":         lr,
    }
    for cls in CLASSES:
        r = ep_metrics["report"].get(str(cls), {})
        log[f"val/f1_c{cls}"]   = r.get("f1-score",  0.0)
        log[f"val/prec_c{cls}"] = r.get("precision", 0.0)
        log[f"val/rec_c{cls}"]  = r.get("recall",    0.0)
    run.log(log)


def _wandb_log_oof(run, oof_metrics, y_true, oof_preds, fold_f1s):
    if run is None:
        return
    import wandb
    # Summary scalars
    run.summary["oof/macro_f1"] = oof_metrics["macro_f1"]
    run.summary["oof/fold_f1_mean"] = float(np.mean(fold_f1s))
    run.summary["oof/fold_f1_std"]  = float(np.std(fold_f1s))
    for cls in CLASSES:
        r = oof_metrics["report"].get(str(cls), {})
        run.summary[f"oof/f1_c{cls}"] = r.get("f1-score", 0.0)

    # Confusion matrix
    run.log({"oof/confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_true.tolist(),
        preds=oof_preds.tolist(),
        class_names=[f"BI-RADS {c}" for c in CLASSES],
    )})

    # Per-class F1 bar chart
    table = wandb.Table(
        columns=["class", "f1", "precision", "recall", "support"],
        data=[
            [
                f"BI-RADS {cls}",
                oof_metrics["report"].get(str(cls), {}).get("f1-score",  0.0),
                oof_metrics["report"].get(str(cls), {}).get("precision", 0.0),
                oof_metrics["report"].get(str(cls), {}).get("recall",    0.0),
                int(oof_metrics["report"].get(str(cls), {}).get("support", 0)),
            ]
            for cls in CLASSES
        ],
    )
    run.log({"oof/per_class": wandb.plot.bar(table, "class", "f1", title="OOF F1 per class")})


# ── Dataset ───────────────────────────────────────────────

class _TextDataset(Dataset):
    """Tokenizes all texts upfront and caches tensors in memory."""

    def __init__(self, texts, tokenizer, max_length, labels=None, sample_weights=None):
        enc = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.token_type_ids = enc.get("token_type_ids")
        self.labels = (
            torch.tensor(labels, dtype=torch.long) if labels is not None else None
        )
        self.sample_weights = (
            torch.tensor(sample_weights, dtype=torch.float32)
            if sample_weights is not None else None
        )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
        }
        if self.token_type_ids is not None:
            item["token_type_ids"] = self.token_type_ids[idx]
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        if self.sample_weights is not None:
            item["sample_weight"] = self.sample_weights[idx]
        return item


# ── Class weights ─────────────────────────────────────────

def _class_weights(y, sample_weights=None, num_classes=None):
    num_classes = int(num_classes or N_CLASSES)
    if sample_weights is None:
        counts = np.bincount(y, minlength=num_classes).astype(float)
    else:
        counts = np.bincount(
            y, weights=np.asarray(sample_weights, dtype=float), minlength=num_classes
        ).astype(float)
    total  = counts.sum()
    w = np.zeros(num_classes)
    present = counts > 0
    w[present] = total / (num_classes * counts[present])
    return torch.tensor(w, dtype=torch.float32)


# ── Optimizer ────────────────────────────────────────────

def _make_sampler(dataset_labels: np.ndarray, num_classes: int | None = None) -> WeightedRandomSampler:
    """Return a WeightedRandomSampler where each sample's weight is 1/class_count."""
    num_classes = int(num_classes or (dataset_labels.max() + 1))
    counts = np.bincount(dataset_labels, minlength=num_classes).astype(float)
    counts = np.where(counts == 0, 1.0, counts)  # avoid division by zero for absent classes
    weights = 1.0 / counts[dataset_labels]
    return WeightedRandomSampler(
        weights=torch.from_numpy(weights).float(),
        num_samples=len(dataset_labels),
        replacement=True,
    )


def _build_optimizer(base_model, lr, weight_decay, llrd_decay=1.0):
    """
    Build AdamW. When llrd_decay < 1.0, applies layer-wise LR decay so lower
    BERT layers (which encode pretrained knowledge) are updated more slowly
    than the classification head. Works for any HuggingFace BERT-family model.

    LR assignment:
      embeddings  : lr * decay^(n_layers)        ← smallest
      layer k     : lr * decay^(n_layers - 1 - k)
      head/pooler : lr                            ← full LR
    """
    no_decay = {"bias", "LayerNorm.weight", "LayerNorm.bias"}

    if llrd_decay >= 1.0:
        # Standard flat-LR AdamW
        decay_params   = [p for n, p in base_model.named_parameters()
                          if p.requires_grad and not any(nd in n for nd in no_decay)]
        nodecay_params = [p for n, p in base_model.named_parameters()
                          if p.requires_grad and     any(nd in n for nd in no_decay)]
        return torch.optim.AdamW(
            [{"params": decay_params,   "lr": lr, "weight_decay": weight_decay},
             {"params": nodecay_params, "lr": lr, "weight_decay": 0.0}]
        )

    # Detect number of transformer layers from parameter names
    layer_indices = set()
    for name, _ in base_model.named_parameters():
        m = re.search(r"\.layer\.(\d+)\.", name)
        if m:
            layer_indices.add(int(m.group(1)))
    n_layers = max(layer_indices) + 1 if layer_indices else 1

    param_groups = []
    for name, param in base_model.named_parameters():
        if not param.requires_grad:
            continue
        wd = 0.0 if any(nd in name for nd in no_decay) else weight_decay
        m = re.search(r"\.layer\.(\d+)\.", name)
        if m:
            k = int(m.group(1))
            group_lr = lr * (llrd_decay ** (n_layers - 1 - k))
        elif "embeddings" in name:
            group_lr = lr * (llrd_decay ** n_layers)
        else:
            group_lr = lr   # head, pooler, classifier
        param_groups.append({"params": [param], "lr": group_lr, "weight_decay": wd})

    return torch.optim.AdamW(param_groups)


# ── Training helpers ──────────────────────────────────────

def _train_epoch(model, loader, optimizer, scaler, scheduler, criterion, device, grad_accum):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        ids   = batch["input_ids"].to(device, non_blocking=True)
        mask  = batch["attention_mask"].to(device, non_blocking=True)
        ttids = batch.get("token_type_ids")
        if ttids is not None:
            ttids = ttids.to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        sample_weight = batch.get("sample_weight")
        if sample_weight is not None:
            sample_weight = sample_weight.to(device, non_blocking=True)

        with autocast(device_type="cuda", enabled=scaler.is_enabled()):
            kwargs = {"input_ids": ids, "attention_mask": mask}
            if ttids is not None:
                kwargs["token_type_ids"] = ttids
            out    = model(**kwargs)
            logits = out.logits if hasattr(out, "logits") else out
            logits = logits.float()

            if sample_weight is None:
                loss = criterion(logits, labels)
            else:
                if isinstance(criterion, _FocalLoss):
                    per_example_loss = criterion(logits, labels, reduction="none")
                else:
                    per_example_loss = F.cross_entropy(
                        logits,
                        labels,
                        weight=criterion.weight,
                        reduction="none",
                        label_smoothing=getattr(criterion, "label_smoothing", 0.0),
                    )
                denom = sample_weight.sum().clamp_min(1e-12)
                loss = (per_example_loss * sample_weight).sum() / denom

            loss = loss / grad_accum

        scaler.scale(loss).backward()
        total_loss += loss.item() * grad_accum

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            params = (model.module if hasattr(model, "module") else model).parameters()
            nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

    return total_loss / len(loader)


@torch.no_grad()
def _predict(model, loader, device, use_amp):
    model.eval()
    all_probs = []
    for batch in loader:
        ids   = batch["input_ids"].to(device, non_blocking=True)
        mask  = batch["attention_mask"].to(device, non_blocking=True)
        ttids = batch.get("token_type_ids")
        if ttids is not None:
            ttids = ttids.to(device, non_blocking=True)

        with autocast(device_type="cuda", enabled=use_amp):
            kwargs = {"input_ids": ids, "attention_mask": mask}
            if ttids is not None:
                kwargs["token_type_ids"] = ttids
            out    = model(**kwargs)
            logits = out.logits if hasattr(out, "logits") else out

        probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)

    return np.concatenate(all_probs, axis=0)


# ── Entry point ───────────────────────────────────────────

def run_training_transformer(config_path: str, notes_override: str = None) -> str:
    with open(config_path) as f:
        config = yaml.safe_load(f)

    exp_name = config["experiment_name"]
    out_dir  = os.path.join("experiments", exp_name)
    os.makedirs(out_dir, exist_ok=True)

    dest = os.path.join(out_dir, "config.yaml")
    if os.path.abspath(config_path) != os.path.abspath(dest):
        shutil.copy(config_path, dest)

    notes = notes_override or config.get("notes", "")

    with run_log(out_dir):
        _run(config, exp_name, out_dir, config_path, notes)

    return out_dir


def _git_hash():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return "unknown"


def _specialist_class_maps(classes):
    classes = [int(c) for c in classes]
    local_to_global = {i: cls for i, cls in enumerate(classes)}
    global_to_local = {cls: i for i, cls in local_to_global.items()}
    return global_to_local, local_to_global


def _expand_probs_to_global(local_probs: np.ndarray, local_to_global: dict[int, int]) -> np.ndarray:
    global_probs = np.zeros((len(local_probs), N_CLASSES), dtype=float)
    for local_idx, global_cls in local_to_global.items():
        global_probs[:, global_cls] = local_probs[:, local_idx]
    return global_probs


def _run(config, exp_name, out_dir, config_path, notes):
    t_start   = time.time()
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    seed       = config.get("seed", 42)
    n_folds    = config["data"]["n_folds"]
    mdl_cfg    = config["model"]
    pretrained      = mdl_cfg["pretrained"]
    max_length      = mdl_cfg.get("max_length", 512)
    p               = mdl_cfg.get("params", {})
    batch_size      = p.get("batch_size", 32)
    eval_batch_size = p.get("eval_batch_size", batch_size * 2)
    grad_accum      = p.get("gradient_accumulation_steps", 1)
    n_epochs        = p.get("epochs", 5)
    lr              = p.get("learning_rate", 2e-5)
    warmup_ratio    = p.get("warmup_ratio", 0.1)
    weight_decay    = p.get("weight_decay", 0.01)
    fp16            = p.get("fp16", True)
    loss_fn                  = p.get("loss_fn", "cross_entropy")   # "cross_entropy" | "focal"
    focal_gamma              = p.get("focal_gamma", 2.0)
    early_stopping_patience  = p.get("early_stopping_patience", None)  # None = disabled
    label_smoothing          = p.get("label_smoothing", 0.0)           # 0.0 = disabled
    llrd_decay               = p.get("llrd_decay", 1.0)               # 1.0 = disabled (flat LR)
    weighted_sampler         = p.get("weighted_sampler", False)        # oversample rare classes per batch

    synth_cfg     = config.get("synthetic_augment", {})
    use_synthetic = synth_cfg.get("enabled", False)
    synth_classes = synth_cfg.get("classes", None)  # None = all classes
    synth_loss_weight = float(synth_cfg.get("loss_weight", 1.0))
    subset_cfg     = config.get("label_subset", {})
    use_label_subset = subset_cfg.get("enabled", False)
    subset_classes = [int(c) for c in subset_cfg.get("classes", [])]
    if use_label_subset and not subset_classes:
        raise ValueError("label_subset.enabled=true requires label_subset.classes")
    if use_label_subset and use_synthetic:
        raise ValueError("synthetic_augment is disabled for label_subset specialist runs in v1")

    gpu_cfg = config.get("gpu", "all")

    torch.manual_seed(seed)
    np.random.seed(seed)

    print(f"\n{'='*55}")
    print(f"Experiment : {exp_name}")
    print(f"Config     : {config_path}")
    print(f"Started    : {timestamp}")
    print(f"Git commit : {_git_hash()}")
    print(f"Pretrained : {pretrained}")
    if notes:
        print(f"Notes      : {notes}")
    if llrd_decay < 1.0:
        print(f"LLRD decay : {llrd_decay}")
    if use_label_subset:
        print(f"Label subset: {subset_classes}")
    print(f"{'='*55}")

    device, gpu_ids = _setup_device(gpu_cfg)
    use_amp = fp16 and torch.cuda.is_available()

    wb_run = _init_wandb(config, exp_name)

    # ── Load data ─────────────────────────────────────────
    print("Loading data...", end=" ", flush=True)
    df = load_train()
    df["text"] = df["report"].apply(clean_text)
    df = make_folds(df, n_folds=n_folds, seed=seed, group_aware=True)
    full_df = df.copy()

    if use_label_subset:
        global_to_local, local_to_global = _specialist_class_maps(subset_classes)
        with open(os.path.join(out_dir, "class_map.json"), "w") as f:
            json.dump(
                {
                    "global_to_local": {str(k): v for k, v in global_to_local.items()},
                    "local_to_global": {str(k): v for k, v in local_to_global.items()},
                },
                f,
                indent=2,
            )
        df_train_src = full_df[full_df["target"].isin(subset_classes)].copy()
        df_train = dedup_for_training(df_train_src)
        df_train["target_local"] = df_train["target"].map(global_to_local).astype(int)
        full_df["target_local"] = full_df["target"].map(global_to_local)
        metric_labels = subset_classes
        model_num_labels = len(subset_classes)
    else:
        df_train = dedup_for_training(full_df)   # one row per unique text, majority-vote labels
        metric_labels = CLASSES
        model_num_labels = N_CLASSES

    # Optional synthetic augmentation — always goes into the training pool,
    # never into OOF validation (which uses `full_ds` = all 18k real rows).
    df_synth = None
    n_synth  = 0
    if use_synthetic:
        df_synth = load_synthetic(classes=synth_classes)
        df_synth["text"] = df_synth["report"].apply(clean_text)
        n_synth = len(df_synth)

    print(
        f"done  ({len(full_df):,} real rows → {len(df_train):,} unique real texts"
        + (f" + {n_synth:,} synthetic" if use_synthetic else "")
        + f" for training, {n_folds} folds)"
    )
    if use_synthetic:
        synth_label = synth_classes if synth_classes is not None else "all"
        print(f"Synthetic classes: {synth_label}")
        print(f"Synthetic loss weight: {synth_loss_weight:g}")

    # Class weights from combined training distribution (real + synthetic if enabled)
    train_labels_for_cw = (
        df_train["target_local"].values if use_label_subset else df_train["target"].values
    )
    train_sample_weights_for_cw = np.ones(len(df_train), dtype=float)
    if df_synth is not None:
        train_labels_for_cw = np.concatenate([train_labels_for_cw, df_synth["target"].values])
        train_sample_weights_for_cw = np.concatenate([
            train_sample_weights_for_cw,
            np.full(len(df_synth), synth_loss_weight, dtype=float),
        ])
    cw = _class_weights(
        train_labels_for_cw,
        train_sample_weights_for_cw,
        num_classes=model_num_labels,
    ).to(device)
    if loss_fn == "focal":
        criterion = _FocalLoss(gamma=focal_gamma, weight=cw)
    else:
        criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=label_smoothing)

    # ── Tokenize once upfront ────────────────────────────
    print(f"Tokenizing with '{pretrained}'...", end=" ", flush=True)
    _, tokenizer = build_model({"pretrained": pretrained, "num_labels": model_num_labels})
    # full_ds: all real rows — used for validation. For specialists, labels are
    # not consumed from the dataset because metrics are computed from full_df.
    full_ds = _TextDataset(
        full_df["text"].tolist(),
        tokenizer,
        max_length,
        labels=np.zeros(len(full_df), dtype=int),
    )
    # train_ds_base: real unique texts for training.
    # If synthetic augmentation is enabled, aug_ds extends this with synthetic rows;
    # synthetic items occupy positions [len(df_train), len(df_train)+n_synth).
    if df_synth is not None:
        aug_texts  = df_train["text"].tolist() + df_synth["text"].tolist()
        real_train_labels = (
            df_train["target_local"].values if use_label_subset else df_train["target"].values
        )
        aug_labels = np.concatenate([real_train_labels, df_synth["target"].values])
        aug_sample_weights = np.concatenate([
            np.ones(len(df_train), dtype=float),
            np.full(len(df_synth), synth_loss_weight, dtype=float),
        ])
        aug_ds = _TextDataset(
            aug_texts,
            tokenizer,
            max_length,
            labels=aug_labels,
            sample_weights=aug_sample_weights,
        )
        dedup_ds = aug_ds   # alias for retrain loop
        synth_positions = list(range(len(df_train), len(df_train) + n_synth))
    else:
        dedup_ds = _TextDataset(
            df_train["text"].tolist(),
            tokenizer,
            max_length,
            labels=(df_train["target_local"].values if use_label_subset else df_train["target"].values),
        )
        synth_positions = []
    print("done")

    oof_probs        = np.zeros((len(full_df), N_CLASSES))
    fold_f1s         = []
    fold_best_epochs = []
    global_epoch     = 0  # monotonically increasing across all folds for wandb x-axis

    # ── OOF cross-validation ──────────────────────────────
    fold_bar = tqdm(range(n_folds), desc="CV folds", unit="fold", ncols=72)
    for fold in fold_bar:
        # Train on deduplicated real rows not in this fold + all synthetic rows;
        # validate on all real rows in this fold (honest OOF — no synthetic in val).
        real_train_pos = df_train.index[df_train["fold"] != fold].tolist()
        train_pos = real_train_pos + synth_positions
        val_pos   = full_df.index[full_df["fold"] == fold].tolist()

        train_ds = Subset(dedup_ds, train_pos)
        val_ds   = Subset(full_ds, val_pos)

        if weighted_sampler:
            real_labels = (
                df_train.loc[df_train["fold"] != fold, "target_local"].values
                if use_label_subset else
                df_train.loc[df_train["fold"] != fold, "target"].values
            )
            fold_labels = (
                np.concatenate([real_labels, df_synth["target"].values])
                if df_synth is not None else real_labels
            )
            sampler = _make_sampler(fold_labels, num_classes=model_num_labels)
            train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                                      num_workers=4, pin_memory=True)
        else:
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                      num_workers=4, pin_memory=True)
        val_loader   = DataLoader(val_ds, batch_size=eval_batch_size, shuffle=False,
                                  num_workers=4, pin_memory=True)

        model, _ = build_model({"pretrained": pretrained, "num_labels": model_num_labels})
        model = model.to(device)
        if len(gpu_ids) > 1:
            model = nn.DataParallel(model, device_ids=gpu_ids)

        steps_per_epoch = max(1, len(train_loader) // grad_accum)
        total_steps  = steps_per_epoch * n_epochs
        warmup_steps = max(1, int(total_steps * warmup_ratio))

        base = model.module if hasattr(model, "module") else model
        optimizer = _build_optimizer(base, lr, weight_decay, llrd_decay)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
        )
        scaler = GradScaler(enabled=use_amp)

        best_f1    = -1.0
        best_probs = None
        best_epoch = 0
        no_improve = 0

        for epoch in range(n_epochs):
            global_epoch += 1
            train_loss = _train_epoch(
                model, train_loader, optimizer, scaler, scheduler,
                criterion, device, grad_accum,
            )
            val_probs_local  = _predict(model, val_loader, device, use_amp)
            val_probs = (
                _expand_probs_to_global(val_probs_local, local_to_global)
                if use_label_subset else val_probs_local
            )
            val_mask = (
                full_df.loc[val_pos, "target"].isin(subset_classes).values
                if use_label_subset else np.ones(len(val_pos), dtype=bool)
            )
            y_val = full_df.loc[val_pos, "target"].values[val_mask]
            ep_metrics = compute_metrics(
                y_val,
                val_probs[val_mask].argmax(1),
                labels=metric_labels,
            )
            ep_f1      = ep_metrics["macro_f1"]
            current_lr = scheduler.get_last_lr()[0]

            fold_bar.set_postfix({
                "fold": f"{fold+1}/{n_folds}",
                "ep":   f"{epoch+1}/{n_epochs}",
                "loss": f"{train_loss:.3f}",
                "F1":   f"{ep_f1:.4f}",
            }, refresh=True)

            _wandb_log_epoch(wb_run, fold + 1, global_epoch, train_loss, ep_metrics, current_lr)

            if ep_f1 > best_f1:
                best_f1    = ep_f1
                best_probs = val_probs.copy()
                best_epoch = epoch + 1
                no_improve = 0
            else:
                no_improve += 1
                if early_stopping_patience is not None and no_improve >= early_stopping_patience:
                    tqdm.write(f"  early stop fold {fold+1} at epoch {epoch+1} "
                               f"(no improvement for {early_stopping_patience} epochs)")
                    break

        for i, pos in enumerate(val_pos):
            oof_probs[pos] = best_probs[i]
        fold_f1s.append(best_f1)
        fold_best_epochs.append(best_epoch)

        if wb_run:
            wb_run.log({"fold_best_f1": best_f1, "fold": fold + 1})

        del model, optimizer, scheduler, scaler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    fold_bar.close()

    # ── OOF evaluation ────────────────────────────────────
    eval_mask = (
        full_df["target"].isin(subset_classes).values
        if use_label_subset else np.ones(len(full_df), dtype=bool)
    )
    oof_preds   = oof_probs.argmax(axis=1)
    oof_metrics = compute_metrics(full_df["target"].values[eval_mask], oof_preds[eval_mask], labels=metric_labels)
    print_metrics(oof_metrics, title=f"OOF Results — {exp_name}", labels=metric_labels)
    print(f"Fold F1s : {[round(f, 4) for f in fold_f1s]}")
    print(f"Mean ± σ : {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")

    oof_df = full_df[["ID", "target", "fold"]].copy()
    oof_df["oof_pred"] = oof_preds
    for i, cls in enumerate(CLASSES):
        oof_df[f"p{cls}"] = oof_probs[:, i]
    oof_df.to_csv(os.path.join(out_dir, "oof_preds.csv"), index=False)
    save_metrics(oof_metrics, out_dir)

    _wandb_log_oof(wb_run, oof_metrics, full_df["target"].values[eval_mask], oof_preds[eval_mask], fold_f1s)

    # ── Optional threshold tuning ─────────────────────────
    if config.get("threshold", {}).get("enabled", False):
        print("\n--- Threshold Tuning ---")
        if use_label_subset:
            raise ValueError("Threshold tuning is disabled for label_subset specialist runs in v1")
        offsets = tune_thresholds(oof_probs, full_df["target"].values, seed=seed)
        np.save(os.path.join(out_dir, "thresholds.npy"), offsets)
        tuned_preds   = oof_probs + offsets
        tuned_metrics = compute_metrics(full_df["target"].values, tuned_preds.argmax(1))
        print_metrics(tuned_metrics, title="OOF Results (after threshold tuning)")
        save_metrics(tuned_metrics, os.path.join(out_dir, "metrics_tuned.json"))
        oof_metrics = tuned_metrics
        if wb_run:
            wb_run.summary["oof/macro_f1_tuned"] = tuned_metrics["macro_f1"]

    # ── Retrain on full data ──────────────────────────────
    # Use mean best epoch from OOF (not fixed n_epochs) so the submission
    # model is trained for the same number of epochs that OOF validated.
    retrain_epochs = max(1, round(sum(fold_best_epochs) / n_folds))
    print(f"\n--- Retraining on full data ({retrain_epochs} epochs, "
          f"mean of OOF best epochs {fold_best_epochs}) ---")
    if weighted_sampler:
        full_labels = train_labels_for_cw   # already includes synthetic if enabled
        full_sampler = _make_sampler(full_labels, num_classes=model_num_labels)
        full_loader = DataLoader(dedup_ds, batch_size=batch_size, sampler=full_sampler,
                                 num_workers=4, pin_memory=True)
    else:
        full_loader = DataLoader(dedup_ds, batch_size=batch_size, shuffle=True,
                                 num_workers=4, pin_memory=True)

    model, tokenizer_full = build_model({"pretrained": pretrained, "num_labels": model_num_labels})
    model = model.to(device)
    if len(gpu_ids) > 1:
        model = nn.DataParallel(model, device_ids=gpu_ids)

    total_steps  = max(1, len(full_loader) // grad_accum) * retrain_epochs
    warmup_steps = max(1, int(total_steps * warmup_ratio))
    base         = model.module if hasattr(model, "module") else model
    optimizer    = _build_optimizer(base, lr, weight_decay, llrd_decay)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    scaler = GradScaler(enabled=use_amp)

    for epoch in tqdm(range(retrain_epochs), desc="Full retrain", unit="ep", ncols=70):
        _train_epoch(model, full_loader, optimizer, scaler, scheduler,
                     criterion, device, grad_accum)

    save_model(model, tokenizer_full, out_dir)
    print(f"Model saved → experiments/{exp_name}/model/")

    if wb_run:
        wb_run.finish()

    duration = time.time() - t_start
    print(f"\nDuration   : {duration:.0f}s  ({duration/60:.1f}m)")
    if use_label_subset:
        print("Specialist label-subset run — skipping append_to_results_log.")
    else:
        append_to_results_log(exp_name, oof_metrics, config,
                              timestamp=timestamp, duration=duration, notes=notes)
    print(f"Outputs    : experiments/{exp_name}/")


In [ ]:
import os, numpy as np, pandas as pd, torch
from torch.amp import autocast
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from src.data import load_test
from src.features import clean_text
from src.train_transformer import _TextDataset

BASE_COMPONENTS  = ['exp023_bertimbau_dedup', 'exp015a_bertimbau_seed7', 'exp015b_bertimbau_seed13', 'exp015c_bertimbau_seed21']
BASE_WEIGHTS     = [0.25, 0.25, 0.25, 0.25]
SPEC023          = 'exp034a_023_specialist'
SPEC456          = 'exp034b_456_specialist'
SPEC023_L2G      = {0: 0, 1: 2, 2: 3}
SPEC456_L2G      = {0: 4, 1: 5, 2: 6}
ALPHA023         = 0.4
ALPHA456         = 0.2
MAX_LENGTH       = 256
N_CLASSES        = 7

def _find_dataset_root():
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'thresholds.npy' in files:
            print(f'Dataset root: {root}'); return root
    raise FileNotFoundError('thresholds.npy not found — attach the model dataset')

def _load_model(dataset_root, name, idx):
    src_dir = os.path.join(dataset_root, name)
    if not os.path.isdir(src_dir):
        cands = [d for d, _, fs in os.walk(dataset_root) if 'config.json' in fs and name in d]
        if not cands: raise FileNotFoundError(f'{name} not found under {dataset_root}')
        src_dir = cands[0]
    link = f'_mdl{idx}'
    if os.path.lexists(link): os.unlink(link)
    os.symlink(os.path.abspath(src_dir), os.path.abspath(link))
    try:
        tok = AutoTokenizer.from_pretrained(link, local_files_only=True)
        mdl = AutoModelForSequenceClassification.from_pretrained(link, local_files_only=True)
    finally:
        os.unlink(link)
    return mdl, tok

def _run_inference(model, tokenizer, texts, device):
    ds = _TextDataset(texts, tokenizer, MAX_LENGTH)
    loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=2)
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            ttids = batch.get('token_type_ids')
            kwargs = {'input_ids': ids, 'attention_mask': mask}
            if ttids is not None: kwargs['token_type_ids'] = ttids.to(device)
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                out = model(**kwargs)
            all_probs.append(torch.softmax(out.logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(all_probs, axis=0)

def _expand_spec_probs(local_probs, local_to_global):
    """Expand (N, K) specialist output to (N, 7) using local→global class map."""
    n = local_probs.shape[0]
    out = np.zeros((n, N_CLASSES), dtype=float)
    for local_idx, global_idx in local_to_global.items():
        out[:, global_idx] = local_probs[:, local_idx]
    return out

def _renorm_group(probs, group):
    sub = probs[:, group].copy()
    denom = sub.sum(axis=1, keepdims=True)
    denom = np.where(denom <= 0, 1.0, denom)
    return sub / denom

def _blend_group(base_probs, spec_probs, group, alpha):
    out = base_probs.copy()
    base_mass = base_probs[:, group].sum(axis=1, keepdims=True)
    blended = (1 - alpha) * _renorm_group(base_probs, group) + alpha * _renorm_group(spec_probs, group)
    out[:, group] = blended * base_mass
    return out

os.chdir('/kaggle/working')
DATASET_ROOT = _find_dataset_root()
test = load_test()
test['text'] = test['report'].apply(clean_text)
print(f'Test rows: {len(test)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Step 1: average base model probs
base_probs = None
for i, (comp, w) in enumerate(zip(BASE_COMPONENTS, BASE_WEIGHTS)):
    print(f'Base model {i+1}/{len(BASE_COMPONENTS)}: {comp}')
    mdl, tok = _load_model(DATASET_ROOT, comp, i)
    mdl = mdl.to(device).eval()
    p = _run_inference(mdl, tok, test['text'].tolist(), device)
    base_probs = p * w if base_probs is None else base_probs + p * w
    del mdl; torch.cuda.empty_cache()
print(f'Base ensemble shape: {base_probs.shape}')

# Step 2: specialist reranking
probs = base_probs.copy()
n_base = len(BASE_COMPONENTS)
if SPEC023:
    print(f'Loading spec023: {SPEC023}')
    mdl, tok = _load_model(DATASET_ROOT, SPEC023, n_base)
    mdl = mdl.to(device).eval()
    spec023_local = _run_inference(mdl, tok, test['text'].tolist(), device)
    spec023_probs = _expand_spec_probs(spec023_local, SPEC023_L2G)
    del mdl; torch.cuda.empty_cache()
    top2 = np.argsort(probs, axis=1)[:, -2:]
    gate023 = np.isin(top2, [0, 2, 3]).all(axis=1)
    print(f'  gate023 activates on {gate023.sum()} rows')
    blended = _blend_group(probs, spec023_probs, [0, 2, 3], ALPHA023)
    probs[gate023] = blended[gate023]
if SPEC456:
    print(f'Loading spec456: {SPEC456}')
    mdl, tok = _load_model(DATASET_ROOT, SPEC456, n_base + 1)
    mdl = mdl.to(device).eval()
    spec456_local = _run_inference(mdl, tok, test['text'].tolist(), device)
    spec456_probs = _expand_spec_probs(spec456_local, SPEC456_L2G)
    del mdl; torch.cuda.empty_cache()
    top2 = np.argsort(probs, axis=1)[:, -2:]
    gate456 = np.isin(top2, [4, 5, 6]).all(axis=1)
    print(f'  gate456 activates on {gate456.sum()} rows')
    blended = _blend_group(probs, spec456_probs, [4, 5, 6], ALPHA456)
    probs[gate456] = blended[gate456]

# Step 3: apply tuned threshold offsets
offsets = np.load(os.path.join(DATASET_ROOT, 'thresholds.npy'))
preds = np.argmax(probs + offsets, axis=1)
print('Applied threshold offsets:', offsets.round(4).tolist())

sub = pd.DataFrame({'ID': test['ID'], 'target': preds})
sub.to_csv('submission.csv', index=False)
print(f'submission.csv: {len(sub)} rows')
print(sub['target'].value_counts().sort_index().to_string())


In [ ]:
import pandas as pd
sub = pd.read_csv('submission.csv')
assert {'ID', 'target'}.issubset(sub.columns)
assert sub['target'].between(0, 6).all(), 'Targets out of range'
print(f'OK — {len(sub)} rows')
sub.head()
